In [2]:
import subprocess
import sys

def check_package(pkg):
    try:
        __import__(pkg)
        print(f"✅ {pkg} is ready")
        return True
    except ImportError:
        print(f"❌ {pkg} missing")
        return False

# Critical packages for Qwen 2.5 + QLoRA
packages = ['transformers', 'torch', 'datasets', 'peft', 'trl', 'accelerate', 'bitsandbytes']
print("--- Checking Kaggle Environment ---")
for p in packages:
    check_package(p)

# Check Transformers version (Qwen 2.5 needs >= 4.40.0)
!python -c "import transformers; print(f'Transformers version: {transformers.__version__}')"

--- Checking Kaggle Environment ---
✅ transformers is ready
✅ torch is ready
✅ datasets is ready
✅ peft is ready
❌ trl missing
✅ accelerate is ready
❌ bitsandbytes missing
Transformers version: 5.0.0


In [3]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 3.9 MB/s eta 0:00:00a 0:00:01


In [4]:
import torch
import transformers
import accelerate
import peft
import trl

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
Transformers version: 5.0.0
CUDA available: True
GPU: Tesla T4
GPU memory: 15.6 GB


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import Dataset

In [6]:
!pip install -q trl
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.5 MB/s eta 0:00:00a 0:00:010m


In [7]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded, parameters:", model.num_parameters())

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded, parameters: 1543714304


In [8]:
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [11]:
import pandas as pd

In [12]:
df=pd.read_csv("/kaggle/input/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/medquad.csv")

In [13]:
df

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [14]:
df = df[['question', 'answer']].dropna()

In [15]:
print("After cleaning:", df.shape)

After cleaning: (16407, 2)


In [16]:
dataset = Dataset.from_pandas(df)

In [17]:
def format_chat(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', '__index_level_0__', 'text'],
        num_rows: 14766
    })
    test: Dataset({
        features: ['question', 'answer', '__index_level_0__', 'text'],
        num_rows: 1641
    })
})


In [18]:
from transformers import DataCollatorForLanguageModeling, Trainer

# Set max sequence length (same as before)
MAX_SEQ_LEN = 1024
tokenizer.model_max_length = MAX_SEQ_LEN
batch_size = 2                    # per_device_train_batch_size
grad_accum = 4
num_epochs=3
warmup_steps = 166
# Tokenization function: tokenize the "text" column and truncate
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,          # we'll use a data collator that pads dynamically
        return_tensors=None,
    )

# Apply tokenization to both splits, removing the original "text" column
tokenized_train = dataset["train"].map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = dataset["test"].map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator for causal language modeling (no masking, just shift labels inside model)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training arguments (unchanged)
training_args = TrainingArguments(
    output_dir="./qwen2.5-lora-output",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_torch",
    max_grad_norm=0.3,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    report_to="none",
)

# Create Trainer (instead of SFTTrainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

Map:   0%|          | 0/14766 [00:00<?, ? examples/s]

Map:   0%|          | 0/1641 [00:00<?, ? examples/s]

In [27]:
trainer.train()
model.save_pretrained("./qwen2.5-lora-final")
tokenizer.save_pretrained("./qwen2.5-lora-final")

Step,Training Loss
10,1.988987
20,1.701033
30,1.545401
40,1.194184
50,1.232955
60,1.189915
70,1.120251
80,1.005387
90,1.138561
100,1.121883


('./qwen2.5-lora-final/tokenizer_config.json',
 './qwen2.5-lora-final/chat_template.jinja',
 './qwen2.5-lora-final/tokenizer.json')

In [28]:
import zipfile
import os

folder = "./qwen2.5-lora-final"
zip_name = "qwen2.5-lora-final.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, folder)
            zipf.write(filepath, arcname)

print("ZIP created:", zip_name)

ZIP created: qwen2.5-lora-final.zip


In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_PATH = "/kaggle/input/datasets/ajoyprasad1998/fine-tuned-llm"

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH
)

model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [21]:
!pip install -q rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 910.6 kB/s eta 0:00:000:00:01


In [22]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd
from tqdm import tqdm
import time

# Use your test set
eval_dataset = dataset["test"]  # Your 1,641 test samples

print(f"Evaluation dataset size: {len(eval_dataset)}")

Evaluation dataset size: 1641


In [23]:
def generate_response(question, model, tokenizer, max_new_tokens=200):
    """Generate a response for a given question"""
    messages = [{"role": "user", "content": question}]
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    response = response[len(formatted):].strip()
    return response

# Test on a subset (or full test set)
test_subset = eval_dataset.select(range(min(100, len(eval_dataset))))

predictions = []
references = []
questions = []

print("Generating predictions...")
for idx, example in enumerate(tqdm(test_subset)):
    # Extract the original question from the formatted text
    # Since we used chat template, we need to parse it back
    # Alternative: store question separately during dataset creation
    question = example["text"].split("user")[-1].split("assistant")[0].strip()
    if not question:
        question = "What is glaucoma?"  # fallback
    
    reference = example["text"].split("assistant")[-1].strip()
    
    pred = generate_response(question, model, tokenizer)
    
    predictions.append(pred)
    references.append(reference)
    questions.append(question)
    
    # Small delay to avoid rate limiting
    time.sleep(0.1)

print(f"Generated {len(predictions)} responses")

Generating predictions...


100%|██████████| 100/100 [24:40<00:00, 14.81s/it]

Generated 100 responses


In [24]:
from rouge_score import rouge_scorer

def compute_rouge(predictions, references):
    """Compute ROUGE-1, ROUGE-2, ROUGE-L scores"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(pred, ref)
        for key in rouge_scores:
            rouge_scores[key].append(scores[key].fmeasure)
    
    results = {}
    for key in rouge_scores:
        results[key] = {
            'mean': np.mean(rouge_scores[key]),
            'std': np.std(rouge_scores[key]),
            'min': np.min(rouge_scores[key]),
            'max': np.max(rouge_scores[key])
        }
    
    return results

rouge_results = compute_rouge(predictions, references)
print("\n=== ROUGE Scores ===")
for metric, scores in rouge_results.items():
    print(f"{metric}: Mean={scores['mean']:.4f} (±{scores['std']:.4f})")


=== ROUGE Scores ===
rouge1: Mean=0.3899 (±0.1595)
rouge2: Mean=0.2131 (±0.2168)
rougeL: Mean=0.2738 (±0.1908)


In [25]:
from bert_score import score as bert_score

def compute_bert_score(predictions, references):
    """Compute BERTScore for semantic similarity"""
    P, R, F1 = bert_score(
        predictions, 
        references, 
        lang='en', 
        model_type='bert-base-uncased',
        verbose=False
    )
    
    return {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item()
    }

bert_results = compute_bert_score(predictions, references)
print("\n=== BERTScore ===")
print(f"Precision: {bert_results['precision']:.4f}")
print(f"Recall: {bert_results['recall']:.4f}")
print(f"F1: {bert_results['f1']:.4f}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== BERTScore ===
Precision: 0.6714
Recall: 0.6038
F1: 0.6337


In [28]:
import torch
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Install bert-score if needed
!pip install -q bert-score
from bert_score import score as bert_score

In [29]:
# Your test set already has question, answer, and text fields
eval_dataset = dataset["test"]

print(f"Evaluation dataset size: {len(eval_dataset)}")

# Extract questions and answers directly
questions = [item["question"] for item in eval_dataset]
references = [item["answer"] for item in eval_dataset]
formatted_texts = [item["text"] for item in eval_dataset]

print(f"Sample question: {questions[0][:100]}...")
print(f"Sample answer: {references[0][:100]}...")

Evaluation dataset size: 1641
Sample question: What are the treatments for Pulmonary alveolar proteinosis acquired ?...
Sample answer: How might acquired pulmonary alveolar proteinosis be treated? The treatment of PAP varies from case ...


In [39]:
def generate_response(question, model, tokenizer, max_new_tokens=300):
    """Generate a response for a given question"""
    try:
        messages = [{"role": "user", "content": question}]
        formatted = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the assistant's response
        response = response[len(formatted):].strip()
        return response
    except Exception as e:
        print(f"Error generating response: {e}")
        return "I don't have an answer for that."

# Test on a subset (adjust based on time)
test_size = len(questions)
predictions = []

print(f"Generating predictions for {test_size} samples...")
for idx in tqdm(range(test_size)):
    question = questions[idx]
    pred = generate_response(question, model, tokenizer)
    predictions.append(pred)
    time.sleep(0.05)  # Small delay to avoid any rate limiting

print(f"Generated {len(predictions)} predictions")

# Use only the subset we evaluated
eval_questions = questions[:test_size]
eval_references = references[:test_size]

Generating predictions for 1641 samples...


100%|██████████| 1641/1641 [8:09:39<00:00, 17.90s/it]  

Generated 1641 predictions


In [40]:
def compute_rouge(predictions, references):
    """Compute ROUGE-1, ROUGE-2, ROUGE-L scores"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    for pred, ref in zip(predictions, references):
        try:
            scores = scorer.score(pred, ref)
            for key in rouge_scores:
                rouge_scores[key].append(scores[key].fmeasure)
        except:
            continue
    
    results = {}
    for key in rouge_scores:
        if rouge_scores[key]:
            results[key] = {
                'mean': np.mean(rouge_scores[key]),
                'std': np.std(rouge_scores[key]),
                'min': np.min(rouge_scores[key]),
                'max': np.max(rouge_scores[key])
            }
        else:
            results[key] = {'mean': 0, 'std': 0, 'min': 0, 'max': 0}
    
    return results

rouge_results = compute_rouge(predictions, eval_references)
print("\n=== ROUGE Scores ===")
for metric, scores in rouge_results.items():
    print(f"{metric}: Mean={scores['mean']:.4f} (±{scores['std']:.4f})")
    print(f"  Min={scores['min']:.4f}, Max={scores['max']:.4f}")


=== ROUGE Scores ===
rouge1: Mean=0.4245 (±0.1960)
  Min=0.0000, Max=0.9777
rouge2: Mean=0.2350 (±0.2485)
  Min=0.0000, Max=0.9776
rougeL: Mean=0.3001 (±0.2275)
  Min=0.0000, Max=0.9777


In [41]:
def compute_bert_score(predictions, references):
    """Compute BERTScore for semantic similarity"""
    try:
        P, R, F1 = bert_score(
            predictions, 
            references, 
            lang='en', 
            model_type='bert-base-uncased',
            verbose=False
        )
        
        return {
            'precision': P.mean().item(),
            'recall': R.mean().item(),
            'f1': F1.mean().item()
        }
    except Exception as e:
        print(f"BERTScore error: {e}")
        return {'precision': 0, 'recall': 0, 'f1': 0}

bert_results = compute_bert_score(predictions, eval_references)
print("\n=== BERTScore ===")
print(f"Precision: {bert_results['precision']:.4f}")
print(f"Recall: {bert_results['recall']:.4f}")
print(f"F1: {bert_results['f1']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== BERTScore ===
Precision: 0.6658
Recall: 0.6535
F1: 0.6572


In [42]:
def compute_perplexity(model, tokenizer, texts, max_samples=20):
    """Compute perplexity on the test set"""
    model.eval()
    losses = []
    
    for text in texts[:max_samples]:
        try:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs, labels=inputs["input_ids"])
                loss = outputs.loss
                losses.append(loss.item())
        except Exception as e:
            print(f"Error computing perplexity: {e}")
            continue
    
    if losses:
        avg_loss = np.mean(losses)
        perplexity = np.exp(avg_loss)
    else:
        avg_loss = 0
        perplexity = 0
    
    return {
        'avg_loss': avg_loss,
        'perplexity': perplexity
    }

perplexity_results = compute_perplexity(model, tokenizer, formatted_texts)
print(f"\n=== Perplexity ===")
print(f"Average Loss: {perplexity_results['avg_loss']:.4f}")
print(f"Perplexity: {perplexity_results['perplexity']:.4f}")


=== Perplexity ===
Average Loss: 0.7735
Perplexity: 2.1674


In [43]:
def measure_inference_speed(model, tokenizer, questions, num_samples=10):
    """Measure tokens per second"""
    times = []
    tokens_generated = []
    
    for question in questions[:num_samples]:
        try:
            messages = [{"role": "user", "content": question}]
            formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
            
            start_time = time.time()
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            end_time = time.time()
            
            generated_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
            tokens_generated.append(generated_tokens)
            times.append(end_time - start_time)
        except Exception as e:
            print(f"Speed test error: {e}")
            continue
    
    if times:
        avg_time = np.mean(times)
        avg_tokens = np.mean(tokens_generated)
        return {
            'avg_time_per_query': avg_time,
            'avg_tokens_generated': avg_tokens,
            'tokens_per_second': avg_tokens / avg_time if avg_time > 0 else 0
        }
    else:
        return {'avg_time_per_query': 0, 'avg_tokens_generated': 0, 'tokens_per_second': 0}

speed_results = measure_inference_speed(model, tokenizer, eval_questions)
print("\n=== Inference Speed ===")
print(f"Avg time per query: {speed_results['avg_time_per_query']:.2f} seconds")
print(f"Avg tokens generated: {speed_results['avg_tokens_generated']:.0f}")
print(f"Tokens per second: {speed_results['tokens_per_second']:.2f}")


=== Inference Speed ===
Avg time per query: 8.06 seconds
Avg tokens generated: 86
Tokens per second: 10.68


In [44]:
print("\n" + "="*70)
print("FINAL EVALUATION SUMMARY")
print("="*70)

print(f"\nModel: Qwen 2.5 1.5B Instruct (Fine-tuned on Medical Q&A)")
print(f"Samples Evaluated: {len(predictions)}")
print(f"Dataset Type: Medical/Health Q&A")

print("\n--- ROUGE Scores ---")
for metric, scores in rouge_results.items():
    print(f"  {metric}: {scores['mean']:.4f} (±{scores['std']:.4f})")

print(f"\n--- BERTScore (Semantic Similarity) ---")
print(f"  Precision: {bert_results['precision']:.4f}")
print(f"  Recall: {bert_results['recall']:.4f}")
print(f"  F1: {bert_results['f1']:.4f}")

print(f"\n--- Perplexity (Model Confidence) ---")
print(f"  {perplexity_results['perplexity']:.4f}")

print(f"\n--- Match Metrics ---")
print(f"  Exact Match: {match_results['exact_match']*100:.2f}%")
print(f"  Partial Match: {match_results['partial_match']*100:.2f}%")

print(f"\n--- Inference Speed ---")
print(f"  {speed_results['tokens_per_second']:.2f} tokens/second")

print("\n" + "="*70)


FINAL EVALUATION SUMMARY

Model: Qwen 2.5 1.5B Instruct (Fine-tuned on Medical Q&A)
Samples Evaluated: 1641
Dataset Type: Medical/Health Q&A

--- ROUGE Scores ---
  rouge1: 0.4245 (±0.1960)
  rouge2: 0.2350 (±0.2485)
  rougeL: 0.3001 (±0.2275)

--- BERTScore (Semantic Similarity) ---
  Precision: 0.6658
  Recall: 0.6535
  F1: 0.6572

--- Perplexity (Model Confidence) ---
  2.1674

--- Match Metrics ---
  Exact Match: 0.00%
  Partial Match: 0.00%

--- Inference Speed ---
  10.68 tokens/second



In [45]:
# Save detailed results
results_df = pd.DataFrame({
    'question': eval_questions,
    'reference': eval_references,
    'prediction': predictions
})

# Add ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(predictions, eval_references):
    try:
        scores = scorer.score(pred, ref)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    except:
        rouge1_scores.append(0)
        rouge2_scores.append(0)
        rougeL_scores.append(0)

results_df['rouge1'] = rouge1_scores
results_df['rouge2'] = rouge2_scores
results_df['rougeL'] = rougeL_scores

# Save to CSV
results_df.to_csv('evaluation_results.csv', index=False)
print("\n✅ Results saved to evaluation_results.csv")

# Show sample predictions
print("\n" + "="*70)
print("SAMPLE PREDICTIONS")
print("="*70)

for i in range(min(3, len(results_df))):
    print(f"\n【Sample {i+1}】")
    print(f"Question: {results_df.iloc[i]['question'][:150]}...")
    print(f"\nReference: {results_df.iloc[i]['reference'][:200]}...")
    print(f"\nPrediction: {results_df.iloc[i]['prediction'][:200]}...")
    print(f"\nROUGE-1: {results_df.iloc[i]['rouge1']:.4f}")
    print("-"*70)


✅ Results saved to evaluation_results.csv

SAMPLE PREDICTIONS

【Sample 1】
Question: What are the treatments for Pulmonary alveolar proteinosis acquired ?...

Reference: How might acquired pulmonary alveolar proteinosis be treated? The treatment of PAP varies from case to case depending upon the age of an affected individual and severity of the disease. Approximately ...

Prediction: d? The treatment of PAP is focused on clearing accumulated material in the lungs and relieving symptoms.  While there are no standard therapies to treat PAP, several different drugs have been tried wi...

ROUGE-1: 0.2520
----------------------------------------------------------------------

【Sample 2】
Question: What is (are) primary ciliary dyskinesia ?...

Reference: Primary ciliary dyskinesia is a disorder characterized by chronic respiratory tract infections, abnormally positioned internal organs, and the inability to have children (infertility). The signs and s...

Prediction: affects the respiratory 

In [49]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

def generate_batch_responses(questions, model, tokenizer, batch_size=8, max_new_tokens=300):
    predictions = []
    total_batches = (len(questions) + batch_size - 1) // batch_size
    print(f"Processing {len(questions)} questions in {total_batches} batches")
    
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_questions = questions[i:i+batch_size]
        
        formatted_batch = []
        for q in batch_questions:
            messages = [{"role": "user", "content": q}]
            formatted = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            formatted_batch.append(formatted)
        
        inputs = tokenizer(
            formatted_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        for j, output in enumerate(outputs):
            response = tokenizer.decode(output, skip_special_tokens=True)
            response = response[len(formatted_batch[j]):].strip()
            predictions.append(response)
    
    return predictions

base_predictions = generate_batch_responses(
    questions=eval_questions,
    model=base_model,
    tokenizer=tokenizer,
    batch_size=16,
    max_new_tokens=300
)

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    for pred, ref in zip(predictions, references):
        try:
            scores = scorer.score(pred, ref)
            for key in rouge_scores:
                rouge_scores[key].append(scores[key].fmeasure)
        except:
            continue
    
    results = {}
    for key in rouge_scores:
        if rouge_scores[key]:
            results[key] = {
                'mean': np.mean(rouge_scores[key]),
                'std': np.std(rouge_scores[key]),
                'min': np.min(rouge_scores[key]),
                'max': np.max(rouge_scores[key])
            }
        else:
            results[key] = {'mean': 0, 'std': 0, 'min': 0, 'max': 0}
    
    return results

base_rouge = compute_rouge(base_predictions, eval_references)

print("\n=== Comparison: Base vs Fine-tuned ===")
print(f"Base model ROUGE-1: {base_rouge['rouge1']['mean']:.4f}")
print(f"Fine-tuned ROUGE-1: {rouge_results['rouge1']['mean']:.4f}")
print(f"Base model ROUGE-2: {base_rouge['rouge2']['mean']:.4f}")
print(f"Fine-tuned ROUGE-2: {rouge_results['rouge2']['mean']:.4f}")
print(f"Base model ROUGE-L: {base_rouge['rougeL']['mean']:.4f}")
print(f"Fine-tuned ROUGE-L: {rouge_results['rougeL']['mean']:.4f}")

rouge1_improvement = ((rouge_results['rouge1']['mean'] - base_rouge['rouge1']['mean']) / base_rouge['rouge1']['mean']) * 100
print(f"\nROUGE-1 Improvement: +{rouge1_improvement:.1f}%")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Processing 1641 questions in 103 batches


100%|██████████| 103/103 [25:14<00:00, 14.70s/it]



=== Comparison: Base vs Fine-tuned ===
Base model ROUGE-1: 0.2790
Fine-tuned ROUGE-1: 0.4245
Base model ROUGE-2: 0.0609
Fine-tuned ROUGE-2: 0.2350
Base model ROUGE-L: 0.1360
Fine-tuned ROUGE-L: 0.3001

ROUGE-1 Improvement: +52.2%
